In [2]:
import matplotlib
import pandas
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
'''
TODO: PatchEmbedding

Break down into patches
(B, C, H, W)
→ (B, C, h, Ph, w, Pw)

Permute:
(B, C, h, Ph, w, Pw)
→ (B, h, w, Ph, Pw, C)

Reshape:
(B, h, w, Ph, Pw, C)
→ (B, h·w, Ph·Pw·C)

Linear projection:
(B, N, patch_dim)
→ (B, N, D)
''';

In [22]:
class PatchEmbedding(nn.Module):
    def __init__(
        self,
        image_size: tuple[int, int],
        patch_size: tuple[int, int],
        channels: int,
        dim: int,
    ) -> None:
        super().__init__()

        if len(image_size) != 2:
            raise ValueError(
                f"Expected 2 dimension image_size but got {len(image_size)}"
            )
        if len(patch_size) != 2:
            raise ValueError(
                f"Expected 2 dimension patch_size but got {len(patch_size)}"
            )
        if not all(x > 0 for x in image_size):
            raise ValueError(
                f"Expected image_size with positive dimensions but got {image_size}"
            )
        if not all(x > 0 for x in patch_size):
            raise ValueError(
                f"Expected patch_size positive dimensions but got {patch_size}"
            )
        if ((image_size[0] % patch_size[0]) != 0) or ((image_size[1] % patch_size[1]) != 0):
            raise ValueError(
                f"Expected image_size divisible by patch_size but got image_size {image_size} and patch_size {patch_size}"
            )
        if not channels > 0:
            raise ValueError(
                f"Expected positive number of channels but got {channels}"
            )
        if not dim > 0:
            raise ValueError(
                f"Expected positive number of dims but got {dim}"
            )

        self.channels = channels
        self.dim = dim
        self.image_height = image_size[0]
        self.image_width = image_size[1]
        self.patch_height = patch_size[0]
        self.patch_width = patch_size[1]
        self.num_patches_h = self.image_height // self.patch_height
        self.num_patches_w = self.image_width // self.patch_width
        self.num_patches = self.num_patches_h * self.num_patches_w
        self.patch_dim = self.patch_height * self.patch_width * channels

        self.project = nn.Linear(
            in_features = self.patch_dim,
            out_features = dim
        )
        
    def forward(
        self,
        images: torch.Tensor,
    ) -> torch.Tensor:
        patches = self.patchify(images)
        embeddings = self.project(patches)
        
        return embeddings

    def patchify(
        self,
        images: torch.Tensor
    ) -> torch.Tensor:
        if images.ndim != 4:
            raise ValueError(
                f"Expected 4 dimension images, but got {images.ndim} dimensions"
            )
        if images.shape[1] != self.channels:
            raise ValueError(
                f"Expected {self.channels} channels but got {images.shape[1]} channels"
            )
        if images.shape[2] != self.image_height:
            raise ValueError(
                f"Expected height {self.image_height} but got {images.shape[2]}"
            )
        if images.shape[3] != self.image_width:
            raise ValueError(
                f"Expected width {self.image_width} but got {images.shape[3]}"
            )

        batch_size = images.shape[0]
            
        images = images.reshape(batch_size, self.channels, self.num_patches_h, self.patch_height, self.num_patches_w, self.patch_width,)
        images = images.permute(0, 2, 4, 3, 5, 1)
        images = images.reshape(batch_size, self.num_patches, self.patch_dim)
        
        return images     

In [27]:
embed = PatchEmbedding(
    [4, 4],
    [2, 2],
    1,
    2
)

sample = torch.tensor(
    [
        [
            [
                [0.0, 1.0, 2.0, 3.0],
                [4.0, 5.0, 6.0, 7.0],
                [8.0, 9.0, 10.0, 11.0],
                [12.0, 13.0, 14.0, 15.0]
            ]
        ]
    ]
)

sample.shape, sample

(torch.Size([1, 1, 4, 4]),
 tensor([[[[ 0.,  1.,  2.,  3.],
           [ 4.,  5.,  6.,  7.],
           [ 8.,  9., 10., 11.],
           [12., 13., 14., 15.]]]]))

In [6]:
sample_emb = embed(sample)
sample_emb.shape, sample_emb

(torch.Size([1, 4, 2]),
 tensor([[[ 0.8337, -1.4565],
          [ 0.2142, -1.0252],
          [-1.6442,  0.2687],
          [-2.2637,  0.7000]]], grad_fn=<ViewBackward0>))

In [7]:
sample_patchify = embed.patchify(sample)
sample_patchify.shape, sample_patchify

(torch.Size([1, 4, 4]),
 tensor([[[ 0.,  1.,  4.,  5.],
          [ 2.,  3.,  6.,  7.],
          [ 8.,  9., 12., 13.],
          [10., 11., 14., 15.]]]))

In [8]:
'''
TODO: ViT
1. Read batch size B
2. Expand CLS from (1, 1, D) to (B, 1, D)
3. Concatenate CLS before the patch tokens
4. Add positional embeddings
5. Return the resulting sequence

patch tokens:      (B, N, D)
batched CLS:       (B, 1, D)
concatenated:      (B, N + 1, D)
position-added:    (B, N + 1, D)
''';

In [25]:
class ViT(nn.Module):
    def __init__(
        self,
        image_size,
        patch_size,
        channels,
        dim,
    ) -> None:
        super().__init__()

        self.patch_embedding = PatchEmbedding(
            image_size,
            patch_size,
            channels,
            dim
        )
        
        self.num_patches = self.patch_embedding.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))

        self.pos_embedding = nn.Embedding(
            self.num_patches + 1,
            dim,
        )

    def forward(
        self,
        images: torch.Tensor
    ) -> torch.Tensor:

        x_emb = self.patch_embedding(images)

        batch_size = images.shape[0]

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)

        x_concat = torch.cat([cls_tokens, x_emb], dim=1)

        positions = torch.arange(start=0, end=self.num_patches + 1, device=x_emb.device).unsqueeze(0)

        out = x_concat + self.pos_embedding(positions)

        return out

In [29]:
model = ViT(
    [4, 4],
    [2, 2],
    1,
    2
)

sequence = model(sample)

assert sequence.shape == (
    sample.shape[0],
    model.num_patches + 1,
    2
)